# simprep 02: variants

Builds the manifest's variants (e.g. IDS R468Q, R468W) from the prepared wild type:
`wt/`, one directory per variant, `variant_record.json` and `variant_report.md`, in a
run directory on Drive (`<drive_root>/<yymm>/run_<yymmdd_hhmmss>/`, with `config.json`).

Two passes, as with `simprep variants`:

1. A manifest without current rotamer decisions gives status `needs_decisions`: the run
   directory holds `manifest.json` with every rotamer candidate as a finding. Decide them
   on the review page (option *Build the side chain in the rotamer named…*), export.
2. Set `manifest` to the exported file and run again (a new run): the variants are built.
   Each variant and its matched wild type are relaxed around the site (OpenMM,
   TASK-006); remaining clashes are reported as warnings.

With `demo = True` the notebook runs pass 1 for 5FQL R468Q / R468W with the committed
test fixture decisions and writes under `googleColab_test/`. It does not decide the
rotamers for you.

In [ ]:
# All configuration lives here (CLAUDE.md, Colab convention 3).
# The manifest must list its variants and, for the build, final rotamer decisions.
demo = True
CONFIG = {
    "repo_url": "https://github.com/dhavirus/proteinMD_pipeline.git",
    "repo_commit": "SET_A_COMMIT_SHA",  # pinned simprep commit to run
    "drive_root": "/content/drive/MyDrive/"
    + ("googleColab_test" if demo else "googleColab_run")
    + "/simprep",
    "work_root": "/content/simprep_work",
    "structure": "/content/drive/MyDrive/simprep_inputs/structure.cif.gz",
    "manifest": "/content/drive/MyDrive/simprep_inputs/manifest.json",
    "run_dir": None,  # set to an existing run directory to resume it
    "seed": 20260926,
}
REPO_DIR = "/content/proteinMD_pipeline"

In [ ]:
import os
import subprocess
import sys

if len(CONFIG["repo_commit"]) != 40:
    raise ValueError("set CONFIG['repo_commit'] to the full 40-character commit SHA to run")
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", CONFIG["repo_url"], REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", CONFIG["repo_commit"]], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", CONFIG["repo_commit"]], check=True)
install = [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[relax]"]
subprocess.run(install, check=True)  # OpenMM for the relaxation step (TASK-006)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import json
from pathlib import Path

import modules.prep_module as prep_module
import modules.variants_module as variants_module

importlib.reload(prep_module)
importlib.reload(variants_module)

if demo:
    import yaml

    from tests.prep_helpers import decided_manifest

    panel = Path(REPO_DIR) / "tests" / "panel"
    fixture = yaml.safe_load((panel / "variant_fixtures" / "5FQL.yaml").read_text())
    prep = yaml.safe_load((panel / "prep_fixtures" / fixture["prep_fixture"]).read_text())
    CONFIG["structure"] = str(panel / fixture["structure"])
    manifest_path = Path("/content/5FQL.variants.manifest.json")
    decided_manifest(panel / fixture["structure"], prep["decisions"], manifest_path)
    document = json.loads(manifest_path.read_text())
    document["variants"] = fixture["variants"]
    manifest_path.write_text(json.dumps(document, indent=2))
    CONFIG["manifest"] = str(manifest_path)

In [ ]:
result = variants_module.run_variants_job(CONFIG)
print(f"{result.status}{' (outputs already complete)' if result.skipped else ''}: {result.run_dir}")
if result.status == "needs_decisions":
    print("Decide the rotamers in", result.run_dir / "manifest.json")
    print("(if that file is absent, decisions_needed.txt says what is missing)")
else:
    print((result.run_dir / "variant_report.md").read_text())